# 004_strategy

**Author:** Wayne Kirk Schmidt  
**Email:** wayne.kirk.schmidt@gmail.com

## Purpose

This stage converts insights from the analysis stage into **tradable strategy signals**.

Using the standardized return data and detected shock events, we construct systematic rules that determine when positions should be opened across the cryptocurrency universe.

The objective is to transform the observed **lead–lag relationships between assets** into a structured trading signal that can be evaluated through backtesting.

---

## Inputs

This notebook reads artifacts generated in earlier stages of the pipeline.

Primary inputs:

- `returns_full.pkl` — synchronized daily return series for all assets  
- `z_scores.pkl` — standardized return series used to identify extreme movements  
- `event_bitmap.pkl` *(optional diagnostic)* — cross-asset shock structure from Stage 3

These datasets are loaded from:
* output/002_enrich/
* output/003_analysis/

---

## Analytical Goals

The strategy construction stage performs three main tasks:

### 1. Detect Shock Events

Identify statistically significant price movements using z-score thresholds.

### 2. Generate Trading Signals

Convert detected shock events into systematic trading signals across the asset universe.

### 3. Construct Position Matrices

Translate signals into time-aligned position matrices that represent the portfolio exposure through time.

These signals form the basis for the simulated trading strategy evaluated in the next stage.

---

## Outputs

Artifacts generated by this stage:

- `signals.pkl` — raw trading signal matrix  
- `positions.pkl` — position matrix representing active portfolio exposure  

These artifacts are written to:
* output/004_strategy/

and registered in the pipeline `manifest.pkl`.

---

## Design Principle

This stage follows several design principles to ensure reproducibility and clarity:

- **Deterministic pipeline** — signals are generated only from artifacts created in earlier stages.
- **Separation of concerns** — strategy construction is separated from backtesting logic.
- **Transparent signal construction** — each step from event detection to portfolio position is explicit.
- **Artifact persistence** — outputs are saved for inspection and reuse.

This structure keeps the strategy layer modular and reproducible within the overall 


### 1. Imports and Environment Setup
### Provide the necessary imports required for to to proceed.   

In [193]:
import pandas as pd
import numpy as np
import pickle
from datetime import datetime, UTC
import math
from pathlib import Path
import matplotlib.pyplot as plt

### 2. Prepare the environment for the notebook

In [194]:
startdate = "2023-01-01"
trading_days = 252
frequency = "1d"

universe = [
    "BTCUSDT",   # Bitcoin
    "ETHUSDT",   # Ethereum
    "BNBUSDT",   # Binance Coin
    "SOLUSDT",   # Solana
    "XRPUSDT",   # Ripple
    "ADAUSDT",   # Cardano
    "DOGEUSDT",  # Dogecoin
    "AVAXUSDT",  # Avalanche
    "LTCUSDT"    # Litecoin
]

execution_delay = [0, 1, 2, 3]
execution_cost_bps = [20, 30, 40]

stage_label = "004_strategy"

OUTPUT_ROOT = Path("../output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE = OUTPUT_ROOT / "manifest.pkl"

DOWNLOAD_DIR = OUTPUT_ROOT / "001_download"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

ENRICH_DIR = OUTPUT_ROOT / "002_enrich"
ENRICH_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_DIR = OUTPUT_ROOT / "003_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

STRATEGY_DIR = OUTPUT_ROOT / "004_strategy"
STRATEGY_DIR.mkdir(parents=True, exist_ok=True)

inspection_window = 20

sigma_threshold = 3

observation_window_length = 10
observation_window = range(1, observation_window_length + 1)

holding_period = 1


### 3. Loading the manifest pickle file

In [195]:
if MANIFEST_FILE.exists():
    manifest = pd.read_pickle(MANIFEST_FILE)
else:
    manifest = {}

manifest.setdefault(stage_label, {})

{'timestamp': '2026-03-20T08:00:55.122642+00:00',
 'artifacts': {'event_response_table': '../output/004_strategy/event_response_table.pkl',
  'leader_follower_summary': '../output/004_strategy/leader_follower_summary.pkl'},
 'rows_event_table': 3,
 'rows_pair_summary': 3}

### 4. Load the pickle files from the previous stage
We use these file contents for our analysis.

In [196]:

# 001_download artifacts
prices = pd.read_pickle(DOWNLOAD_DIR / "PRICES.pkl")
event_panels = {}
for coin in universe:
    panel_file = DOWNLOAD_DIR / f"{coin}.event_panel.pkl"
    event_panels[coin] = pd.read_pickle(panel_file)

# 002_enrich artifacts
returns_full = pd.read_pickle(ENRICH_DIR / "returns_full.pkl")
z_scores = pd.read_pickle(ENRICH_DIR / "z_scores.pkl")
rolling_sigma = pd.read_pickle(ENRICH_DIR / "rolling_sigma.pkl")
price_wide = pd.read_pickle(ENRICH_DIR / "price_wide.pkl")

# 003_analysis artifacts
event_bitmap = pd.read_pickle(ANALYSIS_DIR / "event_bitmap.pkl")
extreme_z_scores = pd.read_pickle(ANALYSIS_DIR / "extreme_z_scores.pkl")
sigma_event_matrix = pd.read_pickle(ANALYSIS_DIR / "sigma_event_matrix.pkl")

print("prices:", prices.shape)
print("returns_full:", returns_full.shape)
print("z_scores:", z_scores.shape)
print("rolling_sigma:", rolling_sigma.shape)
print("price_wide:", price_wide.shape)
print("event_bitmap:", event_bitmap.shape)
print("extreme_z_scores:", extreme_z_scores.shape)
print("sigma_event_matrix:", sigma_event_matrix.shape)
print("event panels loaded:", list(event_panels.keys()))

prices: (10381, 5)
returns_full: (980, 9)
z_scores: (961, 9)
rolling_sigma: (961, 9)
price_wide: (1175, 9)
event_bitmap: (19, 11)
extreme_z_scores: (9, 5)
sigma_event_matrix: (9, 5)
event panels loaded: ['BTCUSDT', 'ETHUSDT', 'BNBUSDT', 'SOLUSDT', 'XRPUSDT', 'ADAUSDT', 'DOGEUSDT', 'AVAXUSDT', 'LTCUSDT']


### 5. Review Shock Events

In [197]:
# count signed sigma values across all coins and days
sigma_counts = sigma_class.stack().value_counts().sort_index()

print(sigma_counts)

-4       4
-3      40
-2     184
-1     888
 0    6259
 1     946
 2     268
 3      58
 4       2
Name: count, dtype: int64


### 6. Explore breakdown of events

In [198]:
# count by signed sigma and coin
pd.set_option("display.width", 2000)
pd.set_option("display.max_columns", None)
stacked = sigma_class.stack()

sigma_coin_counts = (
    stacked
    .groupby([stacked, stacked.index.get_level_values(1)])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

print(sigma_coin_counts)

coin  ADAUSDT  AVAXUSDT  BNBUSDT  BTCUSDT  DOGEUSDT  ETHUSDT  LTCUSDT  SOLUSDT  XRPUSDT
-4          0         0        0        1         0        1        0        0        2
-3          6         3        5        3         3        5        7        3        5
-2         16        19       27       24        21       22       19       18       18
-1        110       108       89       91        91      102       95      108       94
 0        690       689      696      693       709      692      711      665      714
 1        103       106      102      111       101      106       99      129       89
 2         33        31       35       31        28       26       22       33       29
 3          2         5        7        7         8        6        8        5       10
 4          1         0        0        0         0        1        0        0        0


### 7. compute a mask to filter out middle values

In [199]:
extreme_mask = (sigma_coin_counts.index <= -3) | (sigma_coin_counts.index >= 3)

### 8. apply the mask to get number of extreme events per coin

In [200]:
extreme_sigma_coin_counts = sigma_coin_counts.loc[extreme_mask]

print(extreme_sigma_coin_counts)

coin  ADAUSDT  AVAXUSDT  BNBUSDT  BTCUSDT  DOGEUSDT  ETHUSDT  LTCUSDT  SOLUSDT  XRPUSDT
-4          0         0        0        1         0        1        0        0        2
-3          6         3        5        3         3        5        7        3        5
 3          2         5        7        7         8        6        8        5       10
 4          1         0        0        0         0        1        0        0        0


### 9. count up extreme events by sigma

In [201]:
print(extreme_sigma_coin_counts.sum(axis=1))

-4     4
-3    40
 3    58
 4     2
dtype: int64


### 10. count up total events

In [202]:
print(extreme_sigma_coin_counts.sum().sum())

104


### 11. count up total events by coin

In [203]:
coin_extreme_totals = extreme_sigma_coin_counts.sum(axis=0)

print(coin_extreme_totals)

coin
ADAUSDT      9
AVAXUSDT     8
BNBUSDT     12
BTCUSDT     11
DOGEUSDT    11
ETHUSDT     13
LTCUSDT     15
SOLUSDT      8
XRPUSDT     17
dtype: int64


### 12. compute percentage of events per coin of total extreme events

In [204]:
coin_extreme_pct = (coin_extreme_ratio * 100).map(lambda x: f"{x:.2f}%")

print(coin_extreme_pct.sort_values(ascending=False))

coin
ADAUSDT      8.65%
AVAXUSDT     7.69%
SOLUSDT      7.69%
XRPUSDT     16.35%
LTCUSDT     14.42%
ETHUSDT     12.50%
BNBUSDT     11.54%
BTCUSDT     10.58%
DOGEUSDT    10.58%
dtype: object


### 13. validate totals again by cointing up coin totals

In [205]:
total_extremes = coin_extreme_totals.sum()
print(total_extremes)

104


### 14. calculate vector of directions for all of the extreme events of the coins. 
We are looking for normalized f_sum vaues that agree with the direction and have a larger value than the r_coin value

In [206]:
sigma_values = sigma_class.values
dates = sigma_class.index.values
coins = sigma_class.columns.values

n_dates, n_coins = sigma_values.shape

# raw sign matrix
sign_mat = np.sign(sigma_values).astype(int)

# define extreme events
extreme = np.isin(sigma_values, [-4, -3, 3, 4])

# precompute shifted matrices (safe padding with 0)
sign_t0 = sign_mat
sign_t1 = np.vstack([sign_mat[1:], np.zeros((1, n_coins), dtype=int)])
sign_t2 = np.vstack([sign_mat[2:], np.zeros((2, n_coins), dtype=int)])

rows = []

# find all (t_idx, r_idx) where extreme
t_indices, r_indices = np.where(extreme)

for t_idx, r_idx in zip(t_indices, r_indices):

    r_sign = sign_t0[t_idx, r_idx]
    if r_sign == 0:
        continue

    # follower mask (exclude r_idx)
    follower_mask = np.ones(n_coins, dtype=bool)
    follower_mask[r_idx] = False

    f_idx = np.where(follower_mask)[0]

    # vectorized follower extraction
    f0 = sign_t0[t_idx, f_idx]
    f1 = sign_t1[t_idx, f_idx]
    f2 = sign_t2[t_idx, f_idx]

    # sums
    f_sum0 = f0
    f_sum1 = f0 + f1
    f_sum2 = f0 + f1 + f2

    # build rows (still need expansion per follower)
    for i in range(len(f_idx)):
        rows.append([
            dates[t_idx],
            coins[r_idx],
            int(r_sign),
            coins[f_idx[i]],
            int(f0[i]),
            int(f1[i]),
            int(f2[i]),
            int(f_sum0[i]),
            int(f_sum1[i]),
            int(f_sum2[i])
        ])

cols = [
    "date", "r_coin", "r_sign", "f_coin",
    "f_sign0", "f_sign1", "f_sign2",
    "f_sum0", "f_sum1", "f_sum2"
]

event_full_df = pd.DataFrame(rows, columns=cols)
event_full_df

,date,r_coin,r_sign,f_coin,f_sign0,f_sign1,f_sign2,f_sum0,f_sum1,f_sum2
0,2023-08-17,ADAUSDT,-1,AVAXUSDT,-1,1,0,-1,0,0
1,2023-08-17,ADAUSDT,-1,BNBUSDT,-1,0,0,-1,-1,-1
2,2023-08-17,ADAUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2
3,2023-08-17,ADAUSDT,-1,DOGEUSDT,-1,1,0,-1,0,0
4,2023-08-17,ADAUSDT,-1,ETHUSDT,-1,0,0,-1,-1,-1
...,...,...,...,...,...,...,...,...,...,...
827,2026-02-05,XRPUSDT,-1,BTCUSDT,-1,1,0,-1,0,0
828,2026-02-05,XRPUSDT,-1,DOGEUSDT,-1,1,0,-1,0,0
829,2026-02-05,XRPUSDT,-1,ETHUSDT,-1,1,0,-1,0,0
830,2026-02-05,XRPUSDT,-1,LTCUSDT,-1,1,0,-1,0,0


### 15. explore the comparison of f_sum to r_sign

We treat r_sign as the leader’s normalized directional event. 
f_sum0 / f_sum1 / f_sum2 as the follower’s cumulative directional vote across the response window. 
Rows are of interest when the follower cumulative vote has the same 
sign as the leader and a larger absolute value, indicating multi-day directional reinforcement.


In [207]:
event_filtered_df = event_full_df[
    (
        (np.sign(event_full_df["f_sum1"]) == np.sign(event_full_df["r_sign"])) &
        (event_full_df["f_sum1"].abs() > event_full_df["r_sign"].abs())
    ) |
    (
        (np.sign(event_full_df["f_sum2"]) == np.sign(event_full_df["r_sign"])) &
        (event_full_df["f_sum2"].abs() > event_full_df["r_sign"].abs())
    )
].copy()

event_filtered_df

,date,r_coin,r_sign,f_coin,f_sign0,f_sign1,f_sign2,f_sum0,f_sum1,f_sum2
2,2023-08-17,ADAUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2
10,2023-08-17,AVAXUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2
18,2023-08-17,BNBUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2
35,2023-08-17,ETHUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2
43,2023-08-17,LTCUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2
...,...,...,...,...,...,...,...,...,...,...
735,2026-01-05,XRPUSDT,1,SOLUSDT,1,1,-1,1,2,1
747,2026-01-13,ETHUSDT,1,BTCUSDT,1,1,0,1,2,2
756,2026-01-31,BNBUSDT,-1,ETHUSDT,-1,-1,0,-1,-2,-2
758,2026-01-31,BNBUSDT,-1,SOLUSDT,-1,-1,0,-1,-2,-2


### 16. explore the comparison of f_sum to r_sign (continued)

Repeat the same exercise, but this time use booleans for masking and selections later


In [208]:
same_sign_sum1 = np.sign(event_full_df["f_sum1"]) == np.sign(event_full_df["r_sign"])
same_sign_sum2 = np.sign(event_full_df["f_sum2"]) == np.sign(event_full_df["r_sign"])

stronger_sum1 = event_full_df["f_sum1"].abs() > event_full_df["r_sign"].abs()
stronger_sum2 = event_full_df["f_sum2"].abs() > event_full_df["r_sign"].abs()

event_full_df["match_sum1"] = same_sign_sum1 & stronger_sum1
event_full_df["match_sum2"] = same_sign_sum2 & stronger_sum2

event_filtered_df = event_full_df[
    event_full_df["match_sum1"] | event_full_df["match_sum2"]
].copy()

event_filtered_df

,date,r_coin,r_sign,f_coin,f_sign0,f_sign1,f_sign2,f_sum0,f_sum1,f_sum2,match_sum1,match_sum2
2,2023-08-17,ADAUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2,True,True
10,2023-08-17,AVAXUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2,True,True
18,2023-08-17,BNBUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2,True,True
35,2023-08-17,ETHUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2,True,True
43,2023-08-17,LTCUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
735,2026-01-05,XRPUSDT,1,SOLUSDT,1,1,-1,1,2,1,True,False
747,2026-01-13,ETHUSDT,1,BTCUSDT,1,1,0,1,2,2,True,True
756,2026-01-31,BNBUSDT,-1,ETHUSDT,-1,-1,0,-1,-2,-2,True,True
758,2026-01-31,BNBUSDT,-1,SOLUSDT,-1,-1,0,-1,-2,-2,True,True


### 17. Filter on both True for 2 and 3 day sums

In [209]:
event_strong_df = event_full_df[
    event_full_df["match_sum1"] & event_full_df["match_sum2"]
].copy()

event_strong_df

,date,r_coin,r_sign,f_coin,f_sign0,f_sign1,f_sign2,f_sum0,f_sum1,f_sum2,match_sum1,match_sum2
2,2023-08-17,ADAUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2,True,True
10,2023-08-17,AVAXUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2,True,True
18,2023-08-17,BNBUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2,True,True
35,2023-08-17,ETHUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2,True,True
43,2023-08-17,LTCUSDT,-1,BTCUSDT,-1,-1,0,-1,-2,-2,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
714,2025-11-03,SOLUSDT,-1,BNBUSDT,-1,-1,0,-1,-2,-2,True,True
747,2026-01-13,ETHUSDT,1,BTCUSDT,1,1,0,1,2,2,True,True
756,2026-01-31,BNBUSDT,-1,ETHUSDT,-1,-1,0,-1,-2,-2,True,True
758,2026-01-31,BNBUSDT,-1,SOLUSDT,-1,-1,0,-1,-2,-2,True,True


### 18. Assess the size of the population as we provided the successive refinements

In [210]:
len(event_full_df), len(event_filtered_df), len(event_strong_df)


(832, 201, 80)

### 19. Now assess how many 2 day and 3 day count events do we have

In [211]:
event_strong_df["strength"] = event_strong_df["f_sum2"].abs()

In [212]:
event_strong_df["strength"].value_counts()

strength
2    54
3    26
Name: count, dtype: int64

### 20. Count up the number of leader and follower coins in the event space

In [213]:
strong_3 = event_strong_df[event_strong_df["strength"] == 3]

pair_counts = (
    strong_3
    .groupby(["r_coin", "f_coin"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

pair_counts

,r_coin,f_coin,count
23,XRPUSDT,LTCUSDT,2
17,SOLUSDT,AVAXUSDT,2
1,ADAUSDT,ETHUSDT,1
22,XRPUSDT,ETHUSDT,1
21,XRPUSDT,AVAXUSDT,1
20,SOLUSDT,LTCUSDT,1
19,SOLUSDT,ETHUSDT,1
18,SOLUSDT,BTCUSDT,1
16,ETHUSDT,LTCUSDT,1
15,ETHUSDT,BTCUSDT,1


### 21. count up the leader coin frequencies

In [214]:
r_counts = (
    strong_3
    .groupby("r_coin")
    .size()
    .reset_index(name="r_count")
    .sort_values("r_count", ascending=False)
)

r_counts

,r_coin,r_count
4,DOGEUSDT,5
6,SOLUSDT,5
5,ETHUSDT,4
7,XRPUSDT,4
0,ADAUSDT,3
1,AVAXUSDT,2
3,BTCUSDT,2
2,BNBUSDT,1


### 22. count up the follower coin frequencies

In [215]:
f_counts = (
    strong_3
    .groupby("f_coin")
    .size()
    .reset_index(name="f_count")
    .sort_values("f_count", ascending=False)
)

f_counts

,f_coin,f_count
5,LTCUSDT,7
1,AVAXUSDT,6
4,ETHUSDT,5
2,BTCUSDT,3
0,ADAUSDT,2
3,DOGEUSDT,1
6,SOLUSDT,1
7,XRPUSDT,1


### 23. count up the number of events the leader participates in

In [216]:
event_counts = (
    event_full_df
    .groupby("r_coin")
    .size()
    .reset_index(name="total_events")
)

pd.merge(r_counts, event_counts, on="r_coin")

,r_coin,r_count,total_events
0,DOGEUSDT,5,88
1,SOLUSDT,5,64
2,ETHUSDT,4,104
3,XRPUSDT,4,136
4,ADAUSDT,3,72
5,AVAXUSDT,2,64
6,BTCUSDT,2,88
7,BNBUSDT,1,96


### 24. calculate the rate the appears in the total population

In [217]:
leader_stats = pd.merge(r_counts, event_counts, on="r_coin")

leader_stats["r_rate"] = leader_stats["r_count"] / leader_stats["total_events"]

leader_stats = leader_stats.sort_values("r_rate", ascending=False)

leader_stats

,r_coin,r_count,total_events,r_rate
1,SOLUSDT,5,64,0.078125
0,DOGEUSDT,5,88,0.056818
4,ADAUSDT,3,72,0.041667
2,ETHUSDT,4,104,0.038462
5,AVAXUSDT,2,64,0.031250
3,XRPUSDT,4,136,0.029412
6,BTCUSDT,2,88,0.022727
7,BNBUSDT,1,96,0.010417


### 25. Sample leader, looking for the minimum of events in the total_events 

In [218]:
leader_stats["total_events"].min()

64

In [219]:
event_full_path = STRATEGY_DIR / "event_full_df.pkl"
event_filtered_path = STRATEGY_DIR / "event_filtered_df.pkl"
event_strong_path = STRATEGY_DIR / "event_strong_df.pkl"

pair_counts_path = STRATEGY_DIR / "pair_counts.pkl"
leader_stats_path = STRATEGY_DIR / "leader_stats.pkl"

event_full_df.to_pickle(event_full_path)
event_filtered_df.to_pickle(event_filtered_path)
event_strong_df.to_pickle(event_strong_path)

pair_counts.to_pickle(pair_counts_path)
leader_stats.to_pickle(leader_stats_path)

manifest[stage_label] = {
    "timestamp": datetime.now(UTC).isoformat(),
    "artifacts": {
        "event_full_df": str(event_full_path),
        "event_filtered_df": str(event_filtered_path),
        "event_strong_df": str(event_strong_path),
        "pair_counts": str(pair_counts_path),
        "leader_stats": str(leader_stats_path),
    },
    "metrics": {
        "rows_full": len(event_full_df),
        "rows_filtered": len(event_filtered_df),
        "rows_strong": len(event_strong_df),
        "rows_pairs": len(pair_counts),
        "rows_leaders": len(leader_stats),
    }
}

pd.to_pickle(manifest, MANIFEST_FILE)
print("manifest saved:", MANIFEST_FILE)
print("pipeline stages:", sorted(manifest.keys()))
print(f"artifacts in {stage_label}:", sorted(manifest[stage_label]["artifacts"].keys()))

print("\nRow counts:")
print("event_full_df:", len(event_full_df))
print("event_filtered_df:", len(event_filtered_df))
print("event_strong_df:", len(event_strong_df))
print("pair_counts:", len(pair_counts))
print("leader_stats:", len(leader_stats))

manifest saved: ../output/manifest.pkl
pipeline stages: ['001_download', '002_enrich', '003_analysis', '004_strategy']
artifacts in 004_strategy: ['event_filtered_df', 'event_full_df', 'event_strong_df', 'leader_stats', 'pair_counts']

Row counts:
event_full_df: 832
event_filtered_df: 201
event_strong_df: 80
pair_counts: 24
leader_stats: 8
